# Linearized Fisher Updates for Continual Learning

This work studies continual adaptation of deep networks under constrained compute: small batches, limited accelerator memory, and a deliberately small trainable parameter subspace such as a Low Rank Adapter (LoRA) [1]. Elastic Weight Consolidation (EWC) [3] compresses earlier observations into a local quadratic approximation to their log likelihood. At time $t$, we represent that approximation by an anchor and a precision-like information summary,

$$ (\theta_t, \Lambda_t), \qquad \Lambda_t := n_t \widehat{\mathcal I}_t. $$

This pair resembles a Gaussian sufficient statistic, but it is only a local approximation in a general deep network. Its usefulness depends on keeping $\widehat{\mathcal I}_t$ coherent as the model moves through parameter space. A moving average incorporates new curvature observations but lags behind a changing parameter. The proposal here is to reduce that first-order lag with a **linearized Fisher update (LFU)**.

## Definitions and two coupled processes

Let $p_\theta(x)$ be a regular parametric model in a fixed parameter chart $\theta \in \Theta \subseteq \mathbb R^p$. Define

- the log likelihood $\ell(x;\theta) := \log p_\theta(x)$,
- the score $s(x;\theta) := \nabla_\theta \ell(x;\theta)$,
- the Fisher information matrix (FIM) $\mathcal I(\theta) := \mathbb E_\theta[s s^T]$, and
- a parameter displacement $u_t := \theta_{t+1}-\theta_t$.

For a negative log likelihood $L=-\ell$, the loss gradient is $g:=\nabla_\theta L=-s$. For a more general training loss, $g$ defines an empirical-Fisher or pseudo-score construction rather than the model Fisher; the distinction should be made explicit in each experiment.

Two related processes must not be conflated:

1. **Original learning process.** The optimizer, policy update, or adaptation rule uses incoming data to choose $u_t$ and moves $\theta_t$ to $\theta_{t+1}$. This notebook does not prescribe how $u_t$ is chosen; it assumes that the resulting steps are small enough for a local expansion to be useful.
2. **Auxiliary Fisher process.** Given the path produced by the original process, this process estimates the Fisher field along that path and updates $\widehat{\mathcal I}_t$ or $\Lambda_t$. It does not choose the destination. It consumes $u_t$ and asks how the information summary should change because of that move.

In short, the original process decides where the model moves; the auxiliary Fisher process tries to keep the compressed information summary coherent after the move. The auxiliary process is therefore well defined without inventing a Gaussian score family or assuming that it can be placed in canonical coordinates.

## Math sketch: the full Fisher derivative

For any sufficiently regular scalar or matrix-valued function $a(X,\theta)$,

$$ \partial_k \mathbb E_\theta[a(X,\theta)] = \mathbb E_\theta[\partial_k a(X,\theta) + a(X,\theta)s_k(X;\theta)]. $$

Applying this identity to $a=s_i s_j$ gives

$$ \partial_k \mathcal I_{ij}(\theta) = \mathbb E_\theta[(\partial_k s_i)s_j + s_i(\partial_k s_j) + s_i s_j s_k]. $$

Define the Amari-Chentsov tensor [2]

$$ C_{ijk}(\theta) := \mathbb E_\theta[s_i s_j s_k], $$

and define the **residual tensor**

$$ R_{ijk}(\theta) := \mathbb E_\theta[(\partial_k s_i)s_j + s_i(\partial_k s_j)]. $$

Using Einstein summation and $(T:u)_{ij}:=T_{ijk}u^k$, the directional derivative of the Fisher matrix is

$$ D\mathcal I_\theta[u] = (C_\theta+R_\theta):u. $$

The LFU is the first-order Taylor approximation

$$ \boxed{\mathcal I(\theta+u) = \mathcal I(\theta) + (C_\theta+R_\theta):u + O(\|u\|^2).} $$

For a canonical exponential family in its natural coordinates, $\partial_k s_i$ is deterministic and $\mathbb E_\theta[s_j]=0$, so $R_{ijk}=0$ and the Amari-Chentsov term is enough. A deep network in its ordinary weight coordinates is not generally in this case. Gaussianizing an estimator does not supply the unknown map from a model displacement $u$ to a canonical precision displacement, so it does not remove $R$.

The name residual tensor is operational. Strictly, the components $R_{ijk}$ are coordinate dependent under nonlinear reparameterization: they are a combination of connection-coefficient terms, whereas $C_{ijk}$ is an intrinsic tensor. Accordingly, an LFU is a coordinate-local Taylor update, not parallel transport. This is acceptable for the intended implementation, which remains in one fixed deep-network parameter chart or one fixed low-dimensional adapter chart.

## A matrix-free LFU estimator

Let

$$ h_u(X;\theta) := \nabla_\theta^2 \ell(X;\theta)u. $$

Contracting before taking expectations avoids materializing either rank-three array:

$$ D\mathcal I_\theta[u] = \mathbb E_\theta\left[h_u s^T + s h_u^T + (u^T s)ss^T\right]. $$

With the loss convention $L=-\ell$, $g=\nabla L$, and $H_u=\nabla^2 L\,u$, the same identity is

$$ \boxed{D\mathcal I_\theta[u] = \mathbb E_\theta\left[H_u g^T + gH_u^T - (u^T g)gg^T\right].} $$

Thus a sample LFU needs a per-sample gradient, one Hessian-vector product (HVP), and the scalar $u^Tg$. Reverse-mode autodiff computes $H_u$ by differentiating $g^Tu$; no dense Hessian is formed [4]. Moreover, with

$$ U=[g,H_u], \qquad B=\begin{bmatrix}-(u^Tg)&1\\1&0\end{bmatrix}, $$

the sample correction is $UBU^T$ and has rank at most two. An average of these corrections can be accumulated, sketched, truncated, or applied as a linear operator without storing $C$, $R$, or a dense Hessian. The correction is symmetric but need not be positive semidefinite; damping or projection may be needed after a finite first-order step.

## Computational regime

The proposal targets a specific regime:

- Only a manageable parameter subspace is adapted. Full-model LFUs are not assumed practical for large networks.
- Data arrive in small batches, possibly one observation at a time. Per-sample gradients must be retained or vectorized because an outer product of the batch-mean gradient is not the mean of per-sample outer products.
- HVPs are computed by autodiff. Relative to ordinary training, this adds roughly one backward-like pass and retains a higher-order graph, increasing both time and activation memory, but it does not store a $p\times p$ Hessian.
- Fisher summaries and LFU corrections use diagonal, block-diagonal, Kronecker-factored, low-rank, or sketched representations when a dense $p\times p$ matrix is too large.
- The model moves continuously enough that the omitted $O(\|u_t\|^2)$ term is controlled. Large steps should trigger re-estimation, subdivision into smaller LFUs, or rejection of the linear approximation.

Large sample limits may justify separate Gaussian or SDE models of the original learning process, but they are not needed for the LFU identity itself.

## EWC as local information compression

Suppose old observations are summarized at $\theta_t$ by $\Lambda_t=n_t\widehat{\mathcal I}_t$. For a candidate parameter $\vartheta$, their log likelihood is approximated by

$$ \log p(X_{\mathrm{old}};\vartheta) \approx \mathrm{const} - \frac12(\vartheta-\theta_t)^T\Lambda_t(\vartheta-\theta_t). $$

If the new-data likelihood is also approximated near its local MLE $\widehat\theta_{\mathrm{new}}$ with precision $\Lambda_{\mathrm{new}}$, then the EWC-regularized local MLE has the Gaussian product form

$$ \boxed{\widehat\theta_{t+1}=(\Lambda_t+\Lambda_{\mathrm{new}})^{-1}(\Lambda_t\theta_t+\Lambda_{\mathrm{new}}\widehat\theta_{\mathrm{new}}).} $$

The combined precision and natural parameter are

$$ \Lambda_{t+1}=\Lambda_t+\Lambda_{\mathrm{new}}, \qquad \eta_{t+1}=\Lambda_t\theta_t+\Lambda_{\mathrm{new}}\widehat\theta_{\mathrm{new}}. $$

These equations are exact for the two quadratic approximations. They reveal the most economical representation of accumulated quadratic evidence: Gaussian natural parameters $(\Lambda,\eta)$ add. Re-anchoring the same fixed quadratic does not require an LFU and does not change $\Lambda$; its center is recovered by solving $\Lambda\theta=\eta$. In a low-rank or singular trainable subspace, that solve requires damping, a pseudoinverse, or a structured solver.

This exposes an important target distinction. An LFU predicts the **model Fisher field** under the moving model distribution,

$$ \widehat{\mathcal I}_{t\to t+1}^{\mathrm{LFU}} := \widehat{\mathcal I}_t + \widehat{D\mathcal I_{\theta_t}[u_t]}. $$

It does not exactly update the observed information of a fixed old dataset. The latter is $-\nabla^2_\theta\log p(X_{\mathrm{old}};\theta)$, whose change at fixed $X_{\mathrm{old}}$ involves third derivatives of that old-data likelihood rather than $D\mathcal I_\theta[u]$. LFUs are therefore coherent for maintaining an estimate of current local model Fisher, which can shape future EWC factors as the agent traverses the manifold. Using an LFU to evolve the curvature of an already-compressed old-data factor is a further adaptive-EWC approximation and must be tested as such.

## Experimental considerations

LFUs trade moving-average lag for derivative-estimation error. The Amari-Chentsov contribution is a high-variance third score moment, while the residual contribution requires noisy HVPs. Although each contracted sample update is low rank, many updates can accumulate rank and may require truncation. The first-order approximation can also lose positive semidefiniteness or fail when $u_t$ is too large.

The first MNIST experiment should compare four Fisher estimators against frequent re-estimation at the current parameter:

1. an exponential moving average (EMA) baseline,
2. an Amari-Chentsov-only update $C:u_t$,
3. the full LFU $(C+R):u_t$, and
4. periodic fresh Fisher estimates as a higher-compute reference.

Useful ablations include HVP frequency, batch size down to one, adapter dimension, low-rank budget, damping, step size, and subdivision of large steps. Evaluation should measure Fisher approximation error, retained-task performance, adaptation to the changing task, wall-clock time, and peak memory. The central diagnostic is whether the residual tensor materially improves prediction of $\mathcal I(\theta_t+u_t)$ over both EMA and the Amari-Chentsov-only correction.

A second experiment should test whether the same conclusions survive structured or low-rank summaries at a scale closer to the intended edge-robotics setting. A final robotics demonstration can then test whether better-maintained EWC summaries improve continual adaptation of a vision-language model without replaying the original large dataset.

## A model of the original learning process

The LFU construction deliberately leaves $u_t$ unspecified. For numerical experiments, one may separately model the original process as tracking a slowly changing data-generating distribution. Let $P_t$ denote the environment at time $t$, let incoming observations satisfy $X_t\sim P_t$, and let an adaptation rule $A_t$ produce

$$ u_t=A_t(\theta_t,X_t,\text{optimizer state}), \qquad \theta_{t+1}=\theta_t+u_t. $$

This separates three objects: environmental change in $P_t$, the learning rule that determines $u_t$, and the auxiliary Fisher process driven by that realized $u_t$. Model correctness, independence, and slow change can be imposed as experiment-specific assumptions rather than being built into the definition of LFU.

For the theoretically convenient mixture interpretation of EWC, introduce an observed Bernoulli label $M_i$ with $\mathbb P(M_i=1)=\pi$, where $M_i=0$ marks old-task observations and $M_i=1$ marks new-task observations. Expanding the old-data likelihood around its MLE $\theta_t$ gives

$$ \frac1n\log p(X;\vartheta) \approx \frac1n\log p(X^{t+1};\vartheta)-\frac{1-\pi}{2}(\vartheta-\theta_t)^T\mathcal I(\theta_t)(\vartheta-\theta_t)+\mathrm{const}. $$

This recovers the EWC penalty as a local frequentist approximation. It does not make $(\theta_t,n_t\widehat{\mathcal I}_t)$ a globally sufficient statistic, and its accuracy must be checked as the anchor moves.

## Optional diffusion approximation

A diffusion model can be useful for studying the original process, but it is an additional asymptotic model rather than a consequence of LFU. The following display is the fixed-total Bernoulli-composition theory model. For a triangular array with many observations per small update, suppose the conditional increment satisfies

$$ \theta_{k+1,n}-\theta_{k,n}\approx \frac{\pi b(\theta_{k,n})}{n}+\sqrt{\frac{\pi}{n}}\,\mathcal I^{-1/2}(\theta_{k,n})\xi_k, \qquad \xi_k\sim_{iid}\mathcal N(0,I_p). $$

Under the usual regularity, tightness, and Lipschitz assumptions, scaling $k=\lfloor nt\rfloor$ suggests

$$ d\Theta_t=\pi b(\Theta_t)dt+\sqrt{\pi}\,\mathcal I^{-1/2}(\Theta_t)dW_t. $$

Here $b$ and $\pi$ describe how the original process moves. The auxiliary process estimates or predicts the Fisher term along the resulting path. The applied fixed-batch model developed below instead has innovation covariance proportional to $\pi^2/m$ and hence a noise coefficient linear in $\pi$. Experiments should not use either diffusion approximation as evidence that the residual tensor vanishes. The appendix **Diffusion and controlled small-noise limits** gives the triangular-array assumptions, conditional-moment arguments, and time scalings behind both models.

## Why MNIST is comparable to continual reinforcement learning

Both settings can be represented as adaptation to a slowly changing stream. In reinforcement learning, policy updates change the state-action distribution observed by the agent. In the proposed MNIST experiment, the frequency of digit 9 increases gradually after an initial fit on digits 0 through 8. In both cases, the original process follows an optimizer-dependent path $\theta_t$, while the auxiliary process tries to maintain curvature information along that path.

The comparison is intentionally limited: MNIST does not reproduce temporal credit assignment, policy-dependent sampling, or nonstationary transition dynamics. It isolates the narrower question of whether LFUs improve compressed Fisher tracking under controlled distribution shift.


## Single-observation batches

Single-observation updates reduce activation memory and fit the intended online setting, but they do not remove the need for per-sample derivatives. The useful scheduling principle is that each observation corrects the auxiliary Fisher process for the preceding parameter move, then helps the original learning process choose the next move.

### Directional LFU estimator

Suppose the preceding move was $u_{t-1}:=\theta_t-\theta_{t-1}$. After arriving at $\theta_t$, draw $X_t\sim p_{\theta_t}$ and evaluate

$$ s_t:=\nabla_\theta\ell(X_t;\theta_t), \qquad h_t:=\nabla_\theta^2\ell(X_t;\theta_t)u_{t-1}. $$

The lagged single-sample LFU contribution is

$$ \widehat\Delta_t^{\mathrm{lag}}=h_t s_t^T+s_t h_t^T+(u_{t-1}^Ts_t)s_t s_t^T. $$

For a negative log likelihood, let $g_t:=\nabla_\theta L(X_t;\theta_t)$ and $H_t:=\nabla_\theta^2L(X_t;\theta_t)u_{t-1}$. Then

$$ \widehat\Delta_t^{\mathrm{lag}}=H_tg_t^T+g_tH_t^T-(u_{t-1}^Tg_t)g_tg_t^T. $$

### Online LFU recursion

First use the lagged LFU to predict the current Fisher, then blend that prediction with direct curvature evidence at $\theta_t$. In the unified composition paradigm, the same $\pi_t$ that weights new likelihood information in the original learning process is the new-information weight, or forgetting rate, in the auxiliary process:

$$ \widetilde{\mathcal I}_t=\widehat{\mathcal I}_{t-1}+\widehat\Delta_t^{\mathrm{lag}}, $$

$$ \boxed{\widehat{\mathcal I}_t=(1-\pi_t)\widetilde{\mathcal I}_t+\pi_t Z_t}, \qquad Z_t:=s_ts_t^T. $$

The same observation $X_t$ may then help the original learning process choose $u_t$, after which $\theta_{t+1}=\theta_t+u_t$. Thus $X_t$ corrects the Fisher estimate for the past direction $u_{t-1}$ and creates the future direction $u_t$. A large $\pi_t$ both favors the new likelihood in the EWC objective and replaces more of the LFU-predicted Fisher with direct evidence; $1-\pi_t$ is the retained old-information weight. For the conditional calculations below, $\pi_t$ must be predictable with respect to the random quantities it weights. A same-observation plug-in controller is an approximation whose selection bias must be assessed explicitly.

### Lagged directions and recursive error control

Let $\mathcal F_{t-1}$ contain the history after the move to $\theta_t$ but before drawing $X_t$. Then $\theta_t$ and $u_{t-1}$ are $\mathcal F_{t-1}$-measurable. Under conditionally on-model sampling,

$$ \mathbb E\left[\widehat\Delta_t^{\mathrm{lag}}\mid\mathcal F_{t-1}\right]=D\mathcal I_{\theta_t}[u_{t-1}]. $$

This predictability is what removes the same-sample coupling bias. In contrast, pairing $X_t$ with a direction $u_t(X_t)$ generally makes the sample LFU conditionally biased; merely computing the LFU after the optimizer step does not change their shared randomness.

The lagged estimator evaluates the derivative at the endpoint of the preceding move. If $D\mathcal I$ is locally Lipschitz with constant $L$, then

$$ \mathcal I(\theta_t)=\mathcal I(\theta_{t-1})+D\mathcal I_{\theta_t}[u_{t-1}]+r_t, \qquad \|r_t\|\leq\frac{L}{2}\|u_{t-1}\|^2. $$

Define the Fisher estimation error $e_t:=\widehat{\mathcal I}_t-\mathcal I(\theta_t)$, LFU noise $\varepsilon_t:=\widehat\Delta_t^{\mathrm{lag}}-D\mathcal I_{\theta_t}[u_{t-1}]$, and direct-observation noise $\zeta_t:=Z_t-\mathcal I(\theta_t)$. The $\pi_t$-weighted recursion gives

$$ e_t=(1-\pi_t)(e_{t-1}+\varepsilon_t-r_t)+\pi_t\zeta_t. $$

The convex recursion therefore prevents old errors and second-order remainders from accumulating as an uncontracted sum. For constant composition $\pi\in(0,1]$ and uniformly bounded remainder,

$$ \|\mathbb E[e_t]\|\leq(1-\pi)^t\|\mathbb E[e_0]\|+\frac{1-\pi}{\pi}\sup_j\|r_j\|. $$

For step size $\|u_t\|=O(\eta)$, the persistent truncation contribution is consequently $O((1-\pi)\eta^2/\pi)$ rather than an indefinitely growing $O(t\eta^2)$ drift. Recursive weighting controls this error but does not erase it. The composition also mediates a variance tradeoff: $\varepsilon_t$ enters the prediction with coefficient $1-\pi_t$, so an extremely small $\pi_t$ retains LFU noise for many iterations even while it smooths the direct observations $Z_t$. Because $\widehat\Delta_t^{\mathrm{lag}}$ and $Z_t$ use the same $X_t$, their correlation affects variance but not the displayed conditional means when $\pi_t$ is predictable.

These claims require the conditional distribution of $X_t$ to match the distribution defining $\mathcal I(\theta_t)$. Off-model data, uncorrected task drift, or temporal dependence beyond the conditioning state can introduce additional bias; reinforcement-learning experiments will need an appropriate conditional-Fisher or mixing interpretation.

### Effective sample size under adaptive composition

Let $Z_t=m_t^{-1}\sum_{i=1}^{m_t}s_{t,i}s_{t,i}^T$ for a direct batch of size $m_t$; the single-observation case has $m_t=1$. If the recursively weighted contributions were conditionally independent and equally variable after the LFU update, the sum of squared realized weights would obey

$$ q_t=(1-\pi_t)^2q_{t-1}+\frac{\pi_t^2}{m_t}. $$

The corresponding Kish effective size and calibrated information summary are

$$ N_{\mathrm{eff},t}:=q_t^{-1}, \qquad \Lambda_t:=N_{\mathrm{eff},t}\widehat{\mathcal I}_t. $$

For constant $\pi$ and constant $m$, this idealized recursion approaches $N_{\mathrm{eff},\infty}=m(2-\pi)/\pi$. Under adaptive composition, the retained weight of information present immediately after step $s$ is $\prod_{j=s+1}^t(1-\pi_j)$ at step $t$, so there is no separate fixed half-life. This weight ESS is a derived diagnostic, not an additional forgetting hyperparameter. LFU estimation noise and the correlation between $\widehat\Delta_t^{\mathrm{lag}}$ and $Z_t$ are omitted from the simple $q_t$ recursion, so identifying $N_{\mathrm{eff},t}$ with parameter covariance requires the LFU-updated information-summary assumption stated in the appendix and empirical calibration checks.

### Autodiff implementation shape

For each sample, compute $g_t=\nabla L_t$ with a differentiable backward graph, form the scalar $g_t^Tu_{t-1}$, and differentiate that scalar once more to obtain $H_t$. The lagged LFU can then be stored as the two-column factorization

$$ U_t=[g_t,H_t], \qquad \widehat\Delta_t^{\mathrm{lag}}=U_t\begin{bmatrix}-(u_{t-1}^Tg_t)&1\\1&0\end{bmatrix}U_t^T. $$

The first gradient can be detached and reused by the optimizer when choosing $u_t$. This gives the lagged schedule a computational benefit: one observation and one gradient support both the previous LFU correction and the next learning update. In PyTorch, the extra HVP is primarily an additional reverse-mode pass; higher-order autodiff retains the forward graph and creates a graph for the first derivative, so peak memory can rise substantially even though activations are not simply duplicated and no dense Hessian is stored.

Because an LFU is a signed correction, a low-rank implementation should preserve signed factors rather than force every increment into a positive-semidefinite outer product. Positive semidefiniteness is a property to enforce on the resulting Fisher estimate, for example by damping, eigenvalue clipping in the maintained subspace, or a positive structured parameterization.


## Stochastic control: theoretical and applied models

Let $\pi_t\in[0,1]$ regulate adaptation to the population displacement $d\theta_t:=\theta_{t+1}^\star-\theta_t^\star$. In the unified paradigm it is both the new-observation weight in the original learning process and the direct-evidence weight in the auxiliary Fisher recursion. Two local experiments give different noise laws. The first is theoretically convenient and remains useful as an oracle. The second matches the fixed per-step batch size of the intended application and defines the principal controller.

### Theoretical Bernoulli-composition model

Fix a predictable total stencil size $N$ and independently allocate each local likelihood contribution to the new point with probability $\pi_t$. Conditional on an oracle-recentered old summary, the new count is approximately $\pi_tN$, the combined Hessian is $N\mathcal I$, and the update covariance is $\pi_t\mathcal I^{-1}/N$. Writing $T_t:=\operatorname{tr}\mathcal I(\theta_t^\star)^{-1}$, the conditional Euclidean risk is

$$ R_{A,t}(\pi)=(1-\pi)^2\|d\theta_t\|^2+\frac{\pi}{N}T_t, $$

with oracle controller

$$ \boxed{\pi_{A,t}^*=\operatorname{clip}_{[0,1]}\left(1-\frac{T_t}{2N\|d\theta_t\|^2}\right).} $$

At $d\theta_t=0$, define the theoretical oracle by its limiting value $\pi_{A,t}^*=0$ rather than evaluating the displayed quotient.

At the LAN scale $d\theta=b/\sqrt N$, its finite-information small-noise interpolation is

$$ d\Theta_t^{(\varepsilon_N)}=\pi_t b(\Theta_t^{(\varepsilon_N)})dt+\sqrt{\varepsilon_N\pi_t}\,\mathcal I(\Theta_t^{(\varepsilon_N)})^{-1/2}dW_t, \qquad \varepsilon_N=N^{-1/2}. $$

The $\sqrt{\pi_t}$ noise coefficient is the signature of Bernoulli composition: changing $\pi_t$ changes how many new random scores enter a fixed-total experiment. The appendix derives this model rigorously and retains it as the theory oracle.

### Applied fixed-batch EWC model

In the applied experiment, a fixed batch of $m_t$ new observations arrives regardless of $\pi_t$. The compressed old likelihood and mean new likelihood are combined as

$$ Q_t(\vartheta)=(1-\pi_t)Q_{\mathrm{old},t}(\vartheta)+\pi_tQ_{\mathrm{new},t}(\vartheta). $$

Under matched local quadratic curvature, this gives

$$ \widehat\theta_{t+1}\approx(1-\pi_t)\widehat\theta_t+\pi_t\widehat\theta_{\mathrm{new},t}. $$

Let $e_t:=\widehat\theta_t-\theta_t^\star$ and let $\epsilon_{t+1}:=\widehat\theta_{\mathrm{new},t}-\theta_{t+1}^\star$. The applied tracking-error recursion is

$$ \boxed{e_{t+1}=(1-\pi_t)(e_t-d\theta_t)+\pi_t\epsilon_{t+1}.} $$

Assume the LFU-updated summary is covariance calibrated, $e_t$ and $\epsilon_{t+1}$ are locally centered and conditionally independent, and

$$ \tau_{\mathrm{old},t}:=\mathbb E\|e_t\|^2\approx\frac{T_t}{N_{\mathrm{eff},t}}, \qquad \tau_{\mathrm{new},t}:=\mathbb E\|\epsilon_{t+1}\|^2\approx\frac{T_t}{m_t}. $$

The unconditional local tracking risk is then

$$ R_{B,t}(\pi)=(1-\pi)^2\left(\|d\theta_t\|^2+\tau_{\mathrm{old},t}\right)+\pi^2\tau_{\mathrm{new},t}, $$

and its interior minimizer is

$$ \boxed{\pi_{B,t}^*=\frac{\|d\theta_t\|^2+\tau_{\mathrm{old},t}}{\|d\theta_t\|^2+\tau_{\mathrm{old},t}+\tau_{\mathrm{new},t}}.} $$

This fixed-batch rule is the principal applied controller. It is a local bias-variance compromise between a stable EWC estimate and a noisy new-batch estimate, not a post-optimization scaling rule. The implementation sets the EWC odds to $(1-\pi_t)/\pi_t$ and accepts the optimizer solution directly.

For an oracle-recentered fixed-batch increment with $d\theta=b/\sqrt m$, the innovation covariance is $\pi_t^2\mathcal I^{-1}/m$. Taking $h_m=\varepsilon_m=m^{-1/2}$ gives the secondary small-noise interpolation

$$ d\Theta_t^{(\varepsilon_m)}=\pi_t b(\Theta_t^{(\varepsilon_m)})dt+\sqrt{\varepsilon_m}\,\pi_t\mathcal I(\Theta_t^{(\varepsilon_m)})^{-1/2}dW_t. $$

Its Euler covariance over one interval is $\varepsilon_mh_m\pi_t^2\mathcal I^{-1}=\pi_t^2\mathcal I^{-1}/m$. Unlike the Bernoulli model, its noise coefficient is linear in $\pi_t$ because the number of sampled new observations is fixed and their estimator is reweighted. The finite tracking-error recursion above remains the primary applied object because it retains random anchor error explicitly.

### Plug-in trend and covariance estimation

The principal plug-in uses the accepted EWC trajectory rather than a disposable small-batch MLE. The identity

$$ u_t:=\widehat\theta_{t+1}-\widehat\theta_t=d\theta_t+(e_{t+1}-e_t) $$

shows why a single accepted move is not a raw observation of $d\theta_t$, but a local average of accepted moves can estimate the trend when the tracking error is locally stationary. Let $H_p$ be a half-life measured in environmental distance and define

$$ \gamma_t:=1-2^{-\Delta p_t/H_p}, \qquad \widehat d_{t+1\mid t}=(1-\gamma_t)\widehat d_{t\mid t-1}+\gamma_tu_t. $$

For the MNIST experiment, use $H_p=0.20$ by default and check $H_p\in\{0.10,0.20,0.40\}$. Initialize the vector trend at zero. Before one half-life of environmental distance has accumulated, use the zero-trend composition $m_t/(N_{\mathrm{eff},t-1}+m_t)$ rather than a high-variance plug-in decision.

The predictable residual is $r_t:=u_t-\widehat d_{t\mid t-1}$. Under the local fixed-batch model,

$$ \mathbb E\|r_t\|^2\approx a_tT_t, \qquad a_t:=\pi_t^2\left(\frac1{N_{\mathrm{eff},t-1}}+\frac1{m_t}\right). $$

An inversion-free exponentially weighted moment estimator uses the same $\gamma_t$:

$$ V_t=(1-\gamma_t)V_{t-1}+\gamma_t\|r_t\|^2, \qquad A_t=(1-\gamma_t)A_{t-1}+\gamma_ta_t, \qquad \widehat T_t=\frac{V_t}{A_t+\varepsilon}. $$

Then $\widehat\tau_{\mathrm{old},t}=\widehat T_t/N_{\mathrm{eff},t}$ and $\widehat\tau_{\mathrm{new},t}=\widehat T_t/m_{t+1}$. The next controller decision substitutes $\widehat d_{t+1\mid t}$ and these scalar traces into $\pi_{B,t+1}^*$. This estimator avoids both a Fisher inverse and an unregularized small-batch MLE. Its validity requires local trend stability, covariance calibration, and sufficiently weak residual temporal dependence; all three are experimental assumption checks.

When a high-sample reference path supplies the otherwise unavailable population displacement $d\theta_t$, define the oracle-trend residual $r_t^{\mathrm{oracle}}:=u_t-d\theta_t$ and maintain $V_t^{\mathrm{oracle}}=\operatorname{EMA}(\|r_t^{\mathrm{oracle}}\|^2)$ with the same denominator $A_t$. Then $\widehat T_t^{\mathrm{oracle}}=V_t^{\mathrm{oracle}}/(A_t+\varepsilon)$ is still estimated from the accepted parameter series; it does not invert, pseudoinvert, or spectrally regularize an observed Fisher matrix. Comparing $\widehat T_t$ with $\widehat T_t^{\mathrm{oracle}}$ isolates variance attributed to error in the deployable trend estimate. The oracle controller may substitute both $d\theta_t$ and $\widehat T_t^{\mathrm{oracle}}$, while remaining predictable by using only residual moments through step $t-1$.

The ideal controller is predictable: $\pi_t$ is calculated from summaries available through step $t-1$, then the fixed batch at step $t$ is used for both the EWC objective and the Fisher blend. Only after accepting $u_t$ are the trend and trace states updated for $\pi_{t+1}$.


## When local optimality demands forgetting

The one-step rule $\pi_t^*$ need not be a globally desirable learning policy. A large value says that the estimated system change is large relative to local statistical uncertainty. Locally, this favors rapid adaptation. Globally, the same large value also replaces more of the LFU-updated information summary with noisy direct evidence and can erase accumulated curvature information. This coupling is part of the $\pi_t$-centric model rather than an independently tuned effect.

For the online Fisher recursion

$$ \widehat{\mathcal I}_t=(1-\pi_t)\left(\widehat{\mathcal I}_{t-1}+\widehat\Delta_t^{\mathrm{lag}}\right)+\pi_t Z_t, $$

information from an earlier estimate is weighted after $m$ constant-composition steps by approximately $(1-\pi)^m$. For adaptive control its retained weight is the realized product of the intervening $1-\pi_j$ factors. The controller therefore determines its own effective information horizon.

A practical controller bounds the adaptation rate,

$$ \boxed{\pi_{\mathrm{used},t}=\min\left(\pi_{\max},\max(\pi_{\min},\widehat\pi_t^*)\right)}, \qquad 0<\pi_{\min}\leq\pi_{\max}\leq1. $$

The upper bound simultaneously caps parameter adaptation and auxiliary forgetting. This is especially important for $m_t=1$, because $\pi_t$ near one would nearly replace the maintained Fisher with a rank-one observation. The positive lower bound prevents the trace estimating equation from losing all excitation, keeps the EWC odds finite, and lets the agent continue responding to movement after a quiet interval. A deliberate hard freeze remains a separate action outside the ordinary controller. When the unconstrained diagnostic exceeds the upper cap, possible interventions include reducing the optimizer step, subdividing the move into several LFUs, increasing the batch used for curvature estimation, replaying selected observations, lowering the applied composition weight, or freezing part of the trainable subspace.

Under this interpretation, a large $\widehat\pi_t^*$ is an overwhelm diagnostic: the locally optimal tracker would need to adapt too aggressively to preserve one-step accuracy. It is evidence that the system may be outside the regime where a compressed local quadratic and a first-order Fisher update are reliable.


## Citations

[1] Y. Zheng, Y. Zhang, J. van de Weijer, G. M. van de Ven, S. Du, X. Zhang, and Z. Tian, [*Revisiting Weight Regularization for Low-Rank Continual Learning*](https://arxiv.org/abs/2602.17559), arXiv:2602.17559, 2026.

[2] S. Amari and H. Nagaoka, *Methods of Information Geometry*, American Mathematical Society, 2000.

[3] J. Kirkpatrick et al., "Overcoming catastrophic forgetting in neural networks," *Proceedings of the National Academy of Sciences*, 114(13), 3521-3526, 2017.

[4] B. A. Pearlmutter, "Fast Exact Multiplication by the Hessian," *Neural Computation*, 6(1), 147-160, 1994.

[5] S. N. Ethier and T. G. Kurtz, *Markov Processes: Characterization and Convergence*, Wiley, 1986.

[6] H. J. Kushner and G. G. Yin, *Stochastic Approximation and Recursive Algorithms and Applications*, 2nd ed., Springer, 2003.


# Appendix: Diffusion and controlled small-noise limits

This appendix makes precise the scaling arguments for the theoretical Bernoulli-composition model and the applied fixed-batch model. It uses local asymptotic normality (LAN) or an asymptotically linear estimator expansion; it does not place the estimator in an exact Gaussian family, and it does not claim that LFU alone supplies the expansion. Work in one fixed parameter chart $\Theta\subseteq\mathbb R^p$, set $\sigma(\theta):=\mathcal I(\theta)^{-1/2}$, and localize to compact subsets if the coefficients are not globally bounded.

## LFU-updated information-summary assumption

At a current population solution $\theta_{k,N}^\star$, let the compressed old information be $\mathsf S_{k,N}=(\widehat\theta_{k,N},\Lambda_{k,N})$ with $\Lambda_{k,N}=N_{k,N}\widehat{\mathcal I}_{k,N}$. The controller needs a stronger assumption than consistency of the Fisher estimate alone. After the preceding $\pi$-weighted retention and the LFU associated with a local displacement $u_{k,N}$, assume the updated summary is locally calibrated as though it represented $N_{k,N}$ effective observations from the current model:

$$ \widehat{\mathcal I}_{k,N}^{\mathrm{LFU}}=\mathcal I(\theta_{k,N}^\star+u_{k,N})+o_p(1), \qquad \operatorname{Cov}(\widehat\theta_{k,N}\mid\mathcal G_{k,N})=\frac{1}{N_{k,N}}\mathcal I(\theta_{k,N}^\star)^{-1}+o_p(N_{k,N}^{-1}). $$

Here $\mathcal G_{k,N}$ denotes the conditioning information appropriate to the replicated local experiment. The first relation calibrates curvature; the second calibrates estimator uncertainty. Neither relation follows merely from writing $\Lambda=N\widehat{\mathcal I}$, and an accurate LFU does not by itself prove the covariance statement. Their joint use is the **LFU-updated information-summary assumption**. It is a modeling assumption to be checked by the experiment, especially after many updates or large composition weights. The language is deliberately update-based: an LFU is a coordinate-local Taylor update, not a geometric transport.

## Bernoulli composition of a local stencil

Let adjacent population solutions define the stencil displacement

$$ d\theta_{k,N}:=\theta_{k+1,N}^\star-\theta_{k,N}^\star. $$

For a local experiment of predictable size $N$, let $M_{i,k,N}$ be conditionally independent Bernoulli variables with

$$ \mathbb P(M_{i,k,N}=1\mid\mathcal F_{k,N})=\pi_{k,N}. $$

The value $M=0$ allocates a local likelihood contribution to the old point $\theta_{k,N}^\star$, represented computationally by the conditioned EWC summary, while $M=1$ allocates it to a fresh observation from the new point $\theta_{k+1,N}^\star$. Thus $\pi_{k,N}$ is the new-observation composition probability and $K_{k,N}:=\sum_{i=1}^N M_{i,k,N}$ satisfies $K_{k,N}/N\to\pi_{k,N}$ under the corresponding conditional law of large numbers. This is a local statistical construction of the mixed objective; it does not require interpreting fading as literal deletion or survival of stored observations.

## A locally consistent triangular array

For each row index $N$, let $(\mathcal F_{k,N})_{k\geq0}$ be a filtration and let $\xi_{k+1,N}$ be martingale innovations satisfying

$$ \mathbb E[\xi_{k+1,N}\mid\mathcal F_{k,N}]=0, \qquad \mathbb E[\xi_{k+1,N}\xi_{k+1,N}^T\mid\mathcal F_{k,N}]=I_p, $$

together with a conditional Lindeberg condition, uniformly over bounded time intervals. Assume $b$ and $\sigma$ are locally Lipschitz with at most linear growth, so the limiting SDEs and ODEs below are well posed. Let $\pi_{k,N}\in[0,1]$ be $\mathcal F_{k,N}$-measurable. This predictability requirement matters: a control calculated from the same innovation that generates the update generally changes the displayed conditional moments.

A useful common form for the increment is

$$ \Delta\theta_{k,N}=\pi_{k,N}b(\theta_{k,N})h_N+\sqrt{\varepsilon_N\pi_{k,N}h_N}\,\sigma(\theta_{k,N})\xi_{k+1,N}+r_{k,N}. $$

The remainders are required to be negligible at the scale of the accumulated process. For example, on every fixed horizon $T$, it is sufficient that

$$ \sup_{t\leq T}\left\|\sum_{k< t/h_N}r_{k,N}\right\|\xrightarrow{p}0, $$

with analogous $o_p(1)$ control of the accumulated conditional covariance error.

The Bernoulli stencil and LAN make the local moments explicit without assuming an exact Gaussian estimator family. Conditional on the calibrated old summary, its quadratic contributes curvature but no additional score noise. Let $e_{k,N}:=\widehat\theta_{k,N}-\theta_{k,N}^\star$ denote its realized anchor error. Regularity of the likelihood gives, uniformly for $\|e_{k,N}\|+\|d\theta_{k,N}\|=O_p(N^{-1/2})$,

$$ \mathbb E_{\theta^\star+d\theta}[s(X;\widehat\theta)]=\mathcal I(\theta^\star)(d\theta-e)+o_p(N^{-1/2}), \qquad \operatorname{Var}_{\theta^\star+d\theta}[s(X;\widehat\theta)]=\mathcal I(\theta^\star)+o_p(1). $$

Consequently, the $K_{k,N}$ new scores have the LAN expansion

$$ S_{k,N}=K_{k,N}\mathcal I(\theta_{k,N}^\star)(d\theta_{k,N}-e_{k,N})+\sqrt{K_{k,N}}\,\mathcal I(\theta_{k,N}^\star)^{1/2}\xi_{k+1,N}+o_p(\sqrt N). $$

The old quadratic and new likelihood together have local negative Hessian $N\mathcal I(\theta_{k,N}^\star)+o_p(N)$. One Newton step, or an asymptotically equivalent local MLE, therefore satisfies

$$ \widehat\theta_{k+1,N}-\theta_{k,N}^\star=e_{k,N}+\frac{K_{k,N}}{N}(d\theta_{k,N}-e_{k,N})+\frac{\sqrt{K_{k,N}}}{N}\mathcal I(\theta_{k,N}^\star)^{-1/2}\xi_{k+1,N}+o_p(N^{-1/2}). $$

Since $K_{k,N}/N\to\pi_{k,N}$, the conditional mean relative to $\theta_{k,N}^\star$ is $e_{k,N}+\pi_{k,N}(d\theta_{k,N}-e_{k,N})+o_p(N^{-1/2})$ and the conditional covariance is $\pi_{k,N}\mathcal I^{-1}/N+o_p(N^{-1})$. Hence the conditional squared bias relative to the next true solution is $(1-\pi_{k,N})^2\|d\theta_{k,N}-e_{k,N}\|^2$. The simpler controller below sets $e_{k,N}=0$, equivalently using an oracle-recentered current solution. One may instead retain the same algebra by defining its signal as the vector from the realized anchor to the next true solution, $d\theta_{k,N}-e_{k,N}$. Unconditional risk across trajectories must average over anchor uncertainty and any cross-covariance with the new score.

## Ordinary fixed-composition diffusion

Take a fixed $\pi\in[0,1]$, $h_N=N^{-1}$, and $\varepsilon_N=1$. Then

$$ \mathbb E[\Delta\theta_{k,N}\mid\mathcal F_{k,N}]=\pi b(\theta_{k,N})h_N+o_p(h_N), $$

$$ \operatorname{Cov}(\Delta\theta_{k,N}\mid\mathcal F_{k,N})=\pi\mathcal I(\theta_{k,N})^{-1}h_N+o_p(h_N). $$

Let $\Theta^N$ be the piecewise-constant or polygonal interpolation with $k=\lfloor t/h_N\rfloor=\lfloor Nt\rfloor$. The drift characteristics converge to $\int_0^t\pi b(\Theta_s)ds$, the predictable quadratic variation converges to $\int_0^t\pi\mathcal I(\Theta_s)^{-1}ds$, and the conditional Lindeberg condition excludes macroscopic jumps. Standard martingale-problem or diffusion-approximation results [5,6] therefore give

$$ \Theta^N\Rightarrow\Theta \quad\text{in }D([0,T],\mathbb R^p), $$

where the unique weak solution satisfies

$$ \boxed{d\Theta_t=\pi b(\Theta_t)dt+\sqrt\pi\,\mathcal I(\Theta_t)^{-1/2}dW_t.} $$

The factor $\sqrt\pi$ is forced by quadratic variation. For constant $\pi$, the generator is $\mathcal L_\pi=\pi\mathcal L$, so the process is the base learning diffusion run on the slower clock $\pi t$.

## Why stochastic control uses the LAN scale

The one-step controller minimizes mean-squared error to the next population solution, not an error to a thinned dataset. At stencil index $k$, its estimand is

$$ \mathcal R_{k,N}(\pi):=\mathbb E\left[\left\|\widehat\theta_{k+1,N}-(\theta_{k,N}^\star+d\theta_{k,N})\right\|^2\mid\mathcal F_{k,N}\right]. $$

Thus $d\theta_{k,N}$ is concretely the vector from the current true solution point to the next true solution point at the chosen stencil granularity. It is an oracle quantity in the experiment unless a separate estimator is supplied. The displayed risk now adopts the oracle-recentered case $e_{k,N}=0$; otherwise replace $d\theta_{k,N}$ by $d\theta_{k,N}-e_{k,N}$ in its conditional bias term. If the environmental displacement were $b(\theta)/N$, its squared magnitude would be $O(N^{-2})$ while local estimator variance would be $O(N^{-1})$; the asymptotic one-step rule would collapse to no adaptation. If the displacement remained $O(1)$, variance would vanish and the rule would collapse to full adaptation. The nondegenerate LAN modeling assumption is

$$ d\theta_{k,N}=\frac{b(\theta_{k,N}^\star)}{\sqrt N}. $$

Here $b$ is the limiting vector field, in the sense that $\sqrt N\,d\theta_{k,N}\to b(\theta)$ along the triangular array. This scaling is a chosen local asymptotic regime, not a causal relationship between information and environmental movement. Under the conditional LAN expansion above, an update with composition $\pi_{k,N}$ has conditional mean $\pi_{k,N}b/\sqrt N$ and covariance $\pi_{k,N}\mathcal I^{-1}/N$. Its one-step Euclidean risk is

$$ \frac1N\left[(1-\pi)^2\|b(\theta)\|^2+\pi\operatorname{tr}\mathcal I(\theta)^{-1}\right]+o(N^{-1}), $$

so the limiting optimization over $\pi$ is nondegenerate. For known $b$ and interior solutions it gives

$$ \pi^*(\theta)=1-\frac{\operatorname{tr}\mathcal I(\theta)^{-1}}{2\|b(\theta)\|^2}, $$

followed by clipping to $[0,1]$. Equivalently, before substituting $d\theta=b/\sqrt N$, this is the finite-$N$ expression involving $N\|d\theta\|^2$. The formula is conditional on Euclidean loss, the LAN local experiment, calibration of the old summary, and knowledge of the environmental displacement; changing any of these changes the controller.

A time-varying effective size can replace $N$ locally. In particular, $N_{t-1}$ may be used in the one-step formula when it is predictable, diverges along the asymptotic sequence, calibrates the summary covariance as assumed above, and is held fixed while optimizing the next composition $\pi_t$. The associated local alternative is then $d\theta_t=b(\theta_t)/\sqrt{N_{t-1}}+o(N_{t-1}^{-1/2})$. This is a rigorous predictable plug-in construction. It is not the same experiment if choosing $\pi_t$ also changes the denominator through an identity such as $N_t=m_t/\pi_t$ for a fixed new batch size $m_t$; in that case the variance is proportional to $\pi_t^2/m_t$ and the risk must be re-optimized jointly rather than importing the fixed-$N$ formula.

## Controlled fluid limit

Set $h_N=N^{-1/2}$ and suppose the predictable controls $\pi_{k,N}$ converge in a mode sufficient for their drift Riemann sums to converge to a predictable process $\pi_t$. The controlled LAN recursion has the form

$$ \Delta\theta_{k,N}=\pi_{k,N}b(\theta_{k,N})h_N+\sqrt{\pi_{k,N}}\,\sigma(\theta_{k,N})h_N\xi_{k+1,N}+r_{k,N}. $$

Over $\lfloor T/h_N\rfloor=O(\sqrt N)$ steps, the drift is $O(1)$ but the accumulated conditional covariance is only

$$ \sum_{k<T/h_N}\pi_{k,N}\mathcal I(\theta_{k,N})^{-1}h_N^2=O(h_N)=O(N^{-1/2}). $$

A martingale maximal inequality therefore makes the stochastic term vanish uniformly in probability, while the drift converges to its Riemann integral. Subject to the stated remainder and control-convergence assumptions,

$$ \Theta^N\xrightarrow{p}\Theta^0, \qquad \Theta_t^0=\Theta_0^0+\int_0^t\pi_s b(\Theta_s^0)ds, $$

or

$$ \boxed{d\Theta_t^0=\pi_t b(\Theta_t^0)dt.} $$

## Finite-information small-noise diffusion

The deterministic fluid limit suppresses finite-$N$ estimator variability. To retain its leading local covariance, set

$$ \varepsilon_N:=h_N=N^{-1/2} $$

and consider the controlled diffusion

$$ \boxed{d\Theta_t^{(\varepsilon_N)}=\pi_t b(\Theta_t^{(\varepsilon_N)})dt+\sqrt{\varepsilon_N\pi_t}\,\mathcal I(\Theta_t^{(\varepsilon_N)})^{-1/2}dW_t.} $$

Over one Euler interval of length $h_N$, its conditional drift is $\pi_tb h_N$ and its conditional covariance is

$$ \varepsilon_N\pi_t\mathcal I^{-1}h_N=\pi_t\mathcal I^{-1}h_N^2=\frac{\pi_t}{N}\mathcal I^{-1}, $$

which matches the controlled LAN recursion. Equivalently, the diffusion noise coefficient is $O(N^{-1/4})$ and the Brownian increment over one $N^{-1/2}$ interval is also $O(N^{-1/4})$, producing the required $O(N^{-1/2})$ discrete noise. Thus the small-noise SDE is the finite-information diffusion interpolation of the controlled chain, while $\Theta^0$ is its $N\to\infty$ fluid limit.

The ordinary and controlled displays are therefore two regimes of the same local-moment template:

| regime | $h_N$ | $\varepsilon_N$ | accumulated quadratic variation |
|---|---:|---:|---:|
| fixed-composition diffusion | $N^{-1}$ | $1$ | $O(1)$ |
| controlled LAN / small noise | $N^{-1/2}$ | $N^{-1/2}$ | $O(N^{-1/2})$ |

## Fixed-batch stratified local experiment

The Bernoulli construction fixes the total stencil size and lets the number of new observations vary. The applied experiment instead receives a fixed new batch of size $m$ and uses the old summary as a separately represented stratum. Under LAN at the new population point,

$$ \widehat\theta_{\mathrm{new},k,m}=\theta_{k+1,m}^\star+\frac1{\sqrt m}\mathcal I(\theta_{k,m}^\star)^{-1/2}\xi_{k+1,m}+o_p(m^{-1/2}). $$

Locally matched quadratic curvature and the mixture-weighted EWC objective give

$$ \widehat\theta_{k+1,m}=(1-\pi_{k,m})\widehat\theta_{k,m}+\pi_{k,m}\widehat\theta_{\mathrm{new},k,m}+o_p(m^{-1/2}). $$

Writing $e_{k,m}:=\widehat\theta_{k,m}-\theta_{k,m}^\star$ and $d\theta_{k,m}:=\theta_{k+1,m}^\star-\theta_{k,m}^\star$ yields

$$ e_{k+1,m}=(1-\pi_{k,m})(e_{k,m}-d\theta_{k,m})+\frac{\pi_{k,m}}{\sqrt m}\mathcal I(\theta_{k,m}^\star)^{-1/2}\xi_{k+1,m}+o_p(m^{-1/2}). $$

Suppose the old summary is covariance calibrated with effective size $N_{k,m}$, the old error and new innovation are locally centered and conditionally independent, and $d\theta_{k,m}=b(\theta_{k,m}^\star)/\sqrt m$. Then, with $T(\theta):=\operatorname{tr}\mathcal I(\theta)^{-1}$,

$$ m\,\mathbb E\|e_{k+1,m}\|^2=(1-\pi)^2\left(\|b(\theta)\|^2+\frac{m}{N_{k,m}}T(\theta)\right)+\pi^2T(\theta)+o(1). $$

If $m/N_{k,m}$ converges locally, the limiting risk is nondegenerate and has minimizer

$$ \boxed{\pi_B^*(\theta)=\frac{\|b(\theta)\|^2+[m/N_{k,m}]T(\theta)}{\|b(\theta)\|^2+[m/N_{k,m}]T(\theta)+T(\theta)}.} $$

Undoing the LAN substitution gives the finite-step formula in the main text with $\tau_{\mathrm{old}}=T/N_{\mathrm{eff}}$ and $\tau_{\mathrm{new}}=T/m$. This controller is unconditional over calibrated old-anchor error. Conditional on a realized old anchor, replace $d\theta$ by the vector from that anchor to the next true solution and omit $\tau_{\mathrm{old}}$ from the numerator.

For the oracle-recentered innovation, set $h_m=\varepsilon_m=m^{-1/2}$ and write

$$ \Delta\theta_{k,m}=\pi_{k,m}b(\theta_{k,m})h_m+\pi_{k,m}\mathcal I(\theta_{k,m})^{-1/2}h_m\xi_{k+1,m}+r_{k,m}. $$

The matching finite-information interpolation is

$$ \boxed{d\Theta_t^{(\varepsilon_m)}=\pi_t b(\Theta_t^{(\varepsilon_m)})dt+\sqrt{\varepsilon_m}\,\pi_t\mathcal I(\Theta_t^{(\varepsilon_m)})^{-1/2}dW_t.} $$

One Euler interval has covariance $\varepsilon_mh_m\pi_t^2\mathcal I^{-1}=\pi_t^2\mathcal I^{-1}/m$. Its accumulated quadratic variation over $O(\sqrt m)$ controlled LAN steps is $O(m^{-1/2})$, so it has the same deterministic fluid limit as the Bernoulli model but a different finite-information noise law. The applied EWC recursion for $e_{k,m}$ remains more informative than this interpolation because it retains the serial dependence induced by the random anchor.

The two models meet pointwise when a Bernoulli experiment happens to use $m=\pi N$: then $\pi^2/m=\pi/N$. They are nevertheless different control experiments because the applied model holds $m$ fixed while optimizing $\pi$, whereas the theoretical model holds $N$ fixed and lets the new count vary.

| model | fixed quantity | new-score covariance after weighting | small-noise coefficient | role |
|---|---:|---:|---:|---|
| Bernoulli composition | total $N$ | $\pi\mathcal I^{-1}/N$ | $\sqrt{\varepsilon\pi}\,\mathcal I^{-1/2}$ | theory oracle |
| fixed-batch EWC | new batch $m$ | $\pi^2\mathcal I^{-1}/m$ | $\sqrt\varepsilon\,\pi\mathcal I^{-1/2}$ | principal applied model |

This construction still leaves empirical assumptions to validate. In particular, the experiment must test the LFU-updated information-summary calibration, local constancy of the population trend over the chosen half-life, weak enough serial dependence for the residual moment estimator, and fidelity of the optimizer to the local quadratic mixture. Predictable one-step-lagged control removes same-observation selection bias. These are controller assumptions, not consequences of the LFU identity.
